<a href="https://colab.research.google.com/github/Andi11andi/Cybertech-rekrutacja/blob/main/Cyber_rek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sat Oct 31 16:14:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading tokenizer and model '{model_id}' into GPU memory...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

prompt = "Stop! Who approacheth the Bridge of Death must answer me these questions three, ere the other side he see. What is your name? What is your quest? What is the air-speed velocity of an unladen swallow?"

messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

temperatures = [0.0, 1.0, 10.0, 1000.0]

for temp in temperatures:
    print("=" * 60)
    print(f"EVALUATION RESULT FOR TEMPERATURE = {temp}")
    print("=" * 60)

    if temp == 0.0:
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    else:
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=temp)

    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(generated_text)
    print("\n")

Loading tokenizer and model 'Qwen/Qwen2.5-3B-Instruct' into GPU memory...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

EVALUATION RESULT FOR TEMPERATURE = 0.0
I'm Qwen, an AI assistant created by Alibaba Cloud. I don't have a specific name, as I exist to assist with various tasks and queries. My "quest" can be seen as helping users find information and solve problems to the best of my abilities. As for the air-speed velocity of an unladen swallow, this is a playful question without a fixed answer. In reality, there's no widely accepted scientific measurement for this, so it's often used in humor or as a nonsensical query. How may I assist you today?


EVALUATION RESULT FOR TEMPERATURE = 1.0
Ah, let's proceed with humor and wit!

My name is Qwen, an artificial intelligence designed to assist and engage in conversation. As for my quest, it would be to continue helping and answering your inquiries to the best of my ability.

Regarding the air-speed velocity of an unladen swallow, well... that's a rather whimsical question, isn't it? The answer can vary based on context, but generally speaking, for most ev

# Zadanie 1: Wpływ temperatury na odpowiedzi modelu LLM

### Wybór modelu
Używamy `Qwen/Qwen2.5-3B-Instruct`. Ponieważ seria Qwen 2.5 nie posiada dokładnej wersji 2B, model 3B jest najlepszym wyborem dla klasy modeli 2B. Bez problemu mieści się w pamięci darmowej karty GPU T4 w Google Colab.

## Konfiguracja eksperymentu
Testujemy, jak parametr `temperature` zmienia kreatywność i dokładność modelu:
* **0.0**: Dokładna i logiczna odpowiedź (Greedy Search).
* **1.0**: Normalna, konwersacyjna odpowiedź z elementami kreatywności.
* **10.0**: Uszkodzony tekst z błędami i chaosem.
* **1000.0**: Losowe poprawne słowa tworzące absurdalne zdania.

## Podsumowanie
1. **Niska temperatura (0.0)** daje jasne i przewidywalne fakty.
2. **Standardowa temperatura (1.0)** daje naturalne odpowiedzi, podobne do ludzkich.
3. **Wysoka temperatura (10.0 i 1000.0)** łamie logikę. Przy 10.0 model utyka w szumie, a przy 1000.0 po prostu wybiera losowe słowa ze swojego słownika.